In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
pip install timm editdistance scipy -q

In [3]:
import zipfile, os

# ── Configure ALL your zip files and their gt txt files here ──────────────
DRIVE   = '/content/drive/MyDrive/GeoAware_project/datasets'
LOCAL   = '/content/data'

# (zip_path, gt_txt_path, output_txt_path, local_extract_folder)
DATASETS = [


    # TEST
    (f'{DRIVE}/test/art_test.zip',
     f'{DRIVE}/test/art_test_gt.txt',
     f'{LOCAL}/test/art_test_gt.txt',
     f'{LOCAL}/art_test'),

    (f'{DRIVE}/test/iiit5k_test.zip',
     f'{DRIVE}/test/iiit5k_test.txt',
     f'{LOCAL}/test/iiit5k_test.txt',
     f'{LOCAL}/iiit5k_test'),

    (f'{DRIVE}/test/totaltext_test.zip',
     f'{DRIVE}/test/totaltext_test_gt.txt',
     f'{LOCAL}/test/totaltext_test_gt.txt',
     f'{LOCAL}/totaltext_test'),
]
# ──────────────────────────────────────────────────────────────────────────


def find_img_dir(extract_to):
    """Handle zips that extract into a subfolder."""
    entries = os.listdir(extract_to)
    if len(entries) == 1 and os.path.isdir(os.path.join(extract_to, entries[0])):
        return os.path.join(extract_to, entries[0])
    return extract_to


def remap_txt(gt_path, out_path, img_dir):
    remapped = missing = 0
    lines_out = []
    with open(gt_path) as f:
        lines = [l.strip() for l in f if l.strip()]
    for line in lines:
        parts = line.split('\t')
        if len(parts) < 2:
            continue
        filename   = parts[0].replace('\\', '/').split('/')[-1]
        label      = parts[-1].strip()
        local_path = os.path.join(img_dir, filename)
        if os.path.exists(local_path):
            lines_out.append(f"{local_path}\t{label}")
            remapped += 1
        else:
            missing += 1
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, 'w') as f:
        f.write('\n'.join(lines_out))
    return remapped, missing


# ── Main ──────────────────────────────────────────────────────────────────
print("=" * 65)
results = []

for zip_path, gt_path, out_path, extract_to in DATASETS:
    name = os.path.basename(zip_path)

    # Skip missing zips
    if not os.path.exists(zip_path):
        print(f"[SKIP]  {name}  — zip not on Drive")
        results.append((name, None, None))
        continue

    # Unzip (skip if already done)
    os.makedirs(extract_to, exist_ok=True)
    n_existing = len(os.listdir(extract_to))
    if n_existing > 1:
        print(f"[SKIP]  {name}  — already extracted ({n_existing} entries)")
    else:
        print(f"[UNZIP] {name} ...", end=' ', flush=True)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(extract_to)
        print(f"{len(os.listdir(extract_to))} entries")

    # Remap txt
    if not os.path.exists(gt_path):
        print(f"  [SKIP remap] gt txt not found: {gt_path}")
        results.append((name, 0, 0))
        continue

    img_dir = find_img_dir(extract_to)
    remapped, missing = remap_txt(gt_path, out_path, img_dir)
    ok = '✓' if remapped > 0 else '✗'
    print(f"  {ok} remapped={remapped:>6,}  missing={missing:>6,}  → {out_path}")
    results.append((name, remapped, missing))

# Summary
print("\n" + "=" * 65)
print(f"{'Dataset':<35} {'Remapped':>10} {'Missing':>10}")
print("-" * 65)
for name, r, m in results:
    if r is None:
        print(f"{name:<35} {'NOT FOUND':>10}")
    else:
        print(f"{name:<35} {r:>10,} {m:>10,}")
print("=" * 65)

[UNZIP] art_test.zip ... 1 entries
  ✓ remapped= 5,388  missing=     0  → /content/data/test/art_test_gt.txt
[UNZIP] iiit5k_test.zip ... 1 entries
  ✓ remapped= 3,000  missing=     0  → /content/data/test/iiit5k_test.txt
[UNZIP] totaltext_test.zip ... 1 entries
  ✓ remapped= 2,211  missing=     0  → /content/data/test/totaltext_test_gt.txt

Dataset                               Remapped    Missing
-----------------------------------------------------------------
art_test.zip                             5,388          0
iiit5k_test.zip                          3,000          0
totaltext_test.zip                       2,211          0


In [6]:
# ── Cell 2: Paths — EDIT THESE IF YOUR FOLDER NAMES DIFFER ───────────────

PROJECT_ROOT = '/content/drive/MyDrive/GeoAware_project'

# ── Checkpoint ────────────────────────────────────────────────────────────
CKPT_V4 = f'{PROJECT_ROOT}/checkpoints_geoaware_v4/stage3_best.pth'

# ── Existing test annotations (already prepared) ──────────────────────────
DATA = '/content/data/test'
IIIT5K_TXT    = f'{DATA}/iiit5k_test.txt'
ART_TXT       = f'{DATA}/art_test_gt.txt'
TOTALTEXT_TXT = f'{DATA}/totaltext_test_gt.txt'

# ── New datasets on your Windows machine, uploaded to Drive ───────────────
# After running prepare_test_datasets.py locally, upload the 4 .txt files
# to Drive at the path below, OR change these paths to where they are.
TEST_ANNOT_DIR = f'{PROJECT_ROOT}/test_annotations'
SVT_TXT     = f'{TEST_ANNOT_DIR}/svt_test.txt'
ICDAR13_TXT = f'{TEST_ANNOT_DIR}/icdar13_test.txt'
ICDAR15_TXT = f'{TEST_ANNOT_DIR}/icdar15_test.txt'
SCUT_TXT    = f'{TEST_ANNOT_DIR}/scut_test.txt'

import os
print('Checkpoint:')
print(f'  V4-CTC : {"✓" if os.path.exists(CKPT_V4) else "✗ NOT FOUND"} {CKPT_V4}')
print('\nExisting annotations:')
for name, p in [('IIIT5K', IIIT5K_TXT), ('ArT', ART_TXT), ('TotalText', TOTALTEXT_TXT)]:
    print(f'  {name:<12}: {"✓" if os.path.exists(p) else "✗ NOT FOUND"} {p}')
print('\nNew annotations (upload after running prepare_test_datasets.py):')
for name, p in [('SVT', SVT_TXT), ('ICDAR13', ICDAR13_TXT), ('ICDAR15', ICDAR15_TXT), ('SCUT', SCUT_TXT)]:
    print(f'  {name:<12}: {"✓" if os.path.exists(p) else "✗ missing"} {p}')

Checkpoint:
  V4-CTC : ✓ /content/drive/MyDrive/GeoAware_project/checkpoints_geoaware_v4/stage3_best.pth

Existing annotations:
  IIIT5K      : ✓ /content/data/test/iiit5k_test.txt
  ArT         : ✓ /content/data/test/art_test_gt.txt
  TotalText   : ✓ /content/data/test/totaltext_test_gt.txt

New annotations (upload after running prepare_test_datasets.py):
  SVT         : ✓ /content/drive/MyDrive/GeoAware_project/test_annotations/svt_test.txt
  ICDAR13     : ✓ /content/drive/MyDrive/GeoAware_project/test_annotations/icdar13_test.txt
  ICDAR15     : ✓ /content/drive/MyDrive/GeoAware_project/test_annotations/icdar15_test.txt
  SCUT        : ✓ /content/drive/MyDrive/GeoAware_project/test_annotations/scut_test.txt


In [8]:
# ── Cell 3: Prepare NEW annotation files directly in Colab ────────────────
# Run this cell to generate SVT / ICDAR13 / ICDAR15 / SCUT annotation files
# directly from the raw data on your Windows machine.
#
# PREREQUISITE: Upload your raw test datasets to Google Drive first:
#   Drive path: GeoAware_project/datasets/test/
#     ├── ICDAR13_test/           ← folder with images + icdar13_test_gt.json
#     ├── ICDAR15_test_gt.txt     ← GT text file
#     ├── ICDAR15_test/           ← folder with word_*.png images
#     ├── svt_img/                ← SVT images folder
#     │   ├── img/                ← scene images (e.g. 18_01.jpg)
#     │   └── svt_test.xml        ← SVT XML annotation
#     └── scut_test/              ← SCUT images folder
#         └── scut_test.txt       ← SCUT annotation file (can be here or in parent)

import os, re, json, zipfile, xml.etree.ElementTree as ET
from pathlib import Path

RAW_TEST_DIR = f'{PROJECT_ROOT}/datasets/test'
os.makedirs(TEST_ANNOT_DIR, exist_ok=True)

CHARSET_36 = set('0123456789abcdefghijklmnopqrstuvwxyz')

def clean(s):
    return ''.join(c for c in s.lower() if c in CHARSET_36)

def is_valid(s):
    return 1 <= len(clean(s)) <= 25

def bbox_hull(corners):
    xs, ys = [p[0] for p in corners], [p[1] for p in corners]
    return min(xs), min(ys), max(xs), max(ys)

# ── ICDAR 2013 ────────────────────────────────────────────────────────────
def prep_icdar13():
    img_dir   = f'{RAW_TEST_DIR}/ICDAR13_test'
    json_path = f'{img_dir}/icdar13_test_gt.json'
    if not os.path.exists(json_path):
        json_path = f'{RAW_TEST_DIR}/icdar13_test_gt.json'
    if not os.path.exists(json_path):
        print('[ICDAR13] ✗ JSON not found — skipping')
        return 0
    with open(json_path, encoding='utf-8') as f:
        data = json.load(f)
    annots = data.get('annots', data)
    skip   = data.get('unknown', '###')
    kept = skipped = 0
    with open(ICDAR13_TXT, 'w', encoding='utf-8') as out:
        for img_name, info in sorted(annots.items()):
            if not isinstance(info, dict):
                continue
            img_path = f'{img_dir}/{img_name}'
            if not os.path.exists(img_path):
                skipped += len(info.get('text', []))
                continue
            for corners, text in zip(info.get('bbox', []), info.get('text', [])):
                t = text.strip('[](){}')
                if text == skip or not t or not is_valid(t):
                    skipped += 1
                    continue
                x1, y1, x2, y2 = bbox_hull(corners)
                out.write(f'{img_path}|{x1},{y1},{x2},{y2}\t{clean(t)}\n')
                kept += 1
    print(f'[ICDAR13] ✓  kept={kept:,}  skipped={skipped:,}  → {ICDAR13_TXT}')
    return kept

# ── ICDAR 2015 ────────────────────────────────────────────────────────────
def prep_icdar15():
    gt_path  = f'{RAW_TEST_DIR}/ICDAR15_test_gt.txt'
    img_dir  = f'{RAW_TEST_DIR}/ICDAR15_test'
    #zip_path = f'{RAW_TEST_DIR}/ICDAR15.zip'
    if not os.path.exists(gt_path):
        print('[ICDAR15] ✗ GT txt not found — skipping')
        return 0
    if not os.path.isdir(img_dir) and os.path.exists(zip_path):
        print('[ICDAR15] Extracting zip...')
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(RAW_TEST_DIR)
    kept = skipped = 0
    with open(gt_path, encoding='utf-8') as f_in, \
         open(ICDAR15_TXT, 'w', encoding='utf-8') as out:
        for line in f_in:
            line = line.strip()
            if not line:
                continue
            m = re.match(r'^([^,]+),\s*"?([^"]+?)"?\s*$', line)
            if not m:
                skipped += 1
                continue
            img_name, label = m.group(1).strip(), m.group(2).strip()
            if label in ('###', '') or not is_valid(label):
                skipped += 1
                continue
            img_path = f'{img_dir}/{img_name}'
            if not os.path.exists(img_path):
                img_path = f'{RAW_TEST_DIR}/{img_name}'
            if not os.path.exists(img_path):
                skipped += 1
                continue
            out.write(f'{img_path}\t{clean(label)}\n')
            kept += 1
    print(f'[ICDAR15] ✓  kept={kept:,}  skipped={skipped:,}  → {ICDAR15_TXT}')
    return kept

# ── SVT ───────────────────────────────────────────────────────────────────
def prep_svt():
    svt_img_dir = f'{RAW_TEST_DIR}/svt_img'
    xml_path    = f'{svt_img_dir}/svt_test.xml'
    if not os.path.exists(xml_path):
        xml_path = f'{RAW_TEST_DIR}/svt_test.xml'
    if not os.path.exists(xml_path):
        print('[SVT] ✗ svt_test.xml not found — skipping')
        return 0
    tree = ET.parse(xml_path)
    root = tree.getroot()
    kept = skipped = 0
    with open(SVT_TXT, 'w', encoding='utf-8') as out:
        for img_elem in root.iter('image'):
            name_e = img_elem.find('imageName')
            if name_e is None:
                continue
            img_rel  = name_e.text.strip()          # e.g.  img/18_01.jpg
            img_path = f'{svt_img_dir}/{img_rel}'
            if not os.path.exists(img_path):
                img_path = f'{RAW_TEST_DIR}/{img_rel}'
            img_ok = os.path.exists(img_path)
            for rect in img_elem.iter('taggedRectangle'):
                tag_e = rect.find('tag')
                if tag_e is None or not tag_e.text:
                    skipped += 1
                    continue
                text = tag_e.text.strip()
                if text in ('?', '###', '') or not is_valid(text):
                    skipped += 1
                    continue
                if not img_ok:
                    skipped += 1
                    continue
                x  = int(rect.get('x', 0))
                y  = int(rect.get('y', 0))
                x2 = x + int(rect.get('width',  0))
                y2 = y + int(rect.get('height', 0))
                out.write(f'{img_path}|{x},{y},{x2},{y2}\t{clean(text)}\n')
                kept += 1
    print(f'[SVT]     ✓  kept={kept:,}  skipped={skipped:,}  → {SVT_TXT}')
    return kept

# ── SCUT ──────────────────────────────────────────────────────────────────
def prep_scut():
    img_dir  = f'{RAW_TEST_DIR}/scut_test'
    txt_path = f'{RAW_TEST_DIR}/scut_test.txt'
    if not os.path.exists(txt_path):
        txt_path = f'{img_dir}/scut_test.txt'
    if not os.path.exists(txt_path):
        print('[SCUT] ✗ scut_test.txt not found — skipping')
        return 0
    kept = skipped = 0
    with open(txt_path, encoding='utf-8') as f_in, \
         open(SCUT_TXT, 'w', encoding='utf-8') as out:
        for line in f_in:
            parts = line.rstrip('\n').split('\t', 1)
            if len(parts) < 2:
                skipped += 1
                continue
            img_name, phrase = parts[0].strip(), parts[1].strip()
            img_path = f'{img_dir}/{img_name}'
            if not os.path.exists(img_path):
                skipped += 1
                continue
            words = [w for w in phrase.split() if is_valid(w)]
            if not words:
                skipped += 1
                continue
            for w in words:
                out.write(f'{img_path}\t{clean(w)}\n')
                kept += 1
    print(f'[SCUT]    ✓  kept={kept:,}  skipped={skipped:,}  → {SCUT_TXT}')
    return kept

# Run all
print('Preparing test annotations...\n')
n_ic13 = prep_icdar13()
n_ic15 = prep_icdar15()
n_svt  = prep_svt()
n_scut = prep_scut()
print(f'\nDone. Total new samples: {n_ic13+n_ic15+n_svt+n_scut:,}')

Preparing test annotations...

[ICDAR13] ✓  kept=1,081  skipped=14  → /content/drive/MyDrive/GeoAware_project/test_annotations/icdar13_test.txt
[ICDAR15] Extracting zip...
[ICDAR15] ✓  kept=2,077  skipped=0  → /content/drive/MyDrive/GeoAware_project/test_annotations/icdar15_test.txt
[SVT]     ✓  kept=647  skipped=0  → /content/drive/MyDrive/GeoAware_project/test_annotations/svt_test.txt
[SCUT]    ✓  kept=3,188  skipped=191  → /content/drive/MyDrive/GeoAware_project/test_annotations/scut_test.txt

Done. Total new samples: 6,993


In [9]:
# ── Cell 4: Load model ────────────────────────────────────────────────────
import torch, sys, os, time
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from PIL import Image

sys.path.insert(0, PROJECT_ROOT)
for mod in list(sys.modules):
    if 'model' in mod:
        del sys.modules[mod]

import timm
_orig = timm.create_model
timm.create_model = lambda n, **kw: _orig(n, **{**kw, 'pretrained': False})
import models.model as _mm
_mm.CHARSET   = _mm.CHARSET_36
_mm.BLANK_IDX = len(_mm.CHARSET_36)
from models.model import PARSeqGeoAware, improved_ctc_decode, CHARSET, BLANK_IDX
timm.create_model = _orig

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CHARSET_SET = set(CHARSET)
print(f'Device: {DEVICE}')

# Load V4 CTC
ckpt_v4 = torch.load(CKPT_V4, map_location='cpu')
sd_v4   = ckpt_v4['model_state_dict']
model_v4 = PARSeqGeoAware(
    num_chars=sd_v4['head.weight'].shape[0],
    use_geometric=True, use_rectification=True,
    use_tps=False, use_attention=False, max_len=25)
model_v4.load_state_dict(sd_v4, strict=False)
model_v4.eval().to(DEVICE)
print(f'✓ V4-CTC loaded  epoch={ckpt_v4.get("epoch","?")}  '
      f'loss={ckpt_v4.get("ctc_loss",0):.4f}')

# Transforms
_T = T.Compose([T.Resize((64,256)), T.Grayscale(1),
                T.ToTensor(), T.Normalize([0.5],[0.5])])

def load_img(path_spec):
    """Load image; crop bbox if path contains |x1,y1,x2,y2."""
    if '|' in path_spec:
        img_p, bb = path_spec.split('|', 1)
        x1, y1, x2, y2 = map(int, bb.split(','))
        img = Image.open(img_p).convert('RGB')
        w, h = img.size
        x1, y1 = max(0,x1), max(0,y1)
        x2, y2 = min(w,x2), min(h,y2)
        if x2 > x1 and y2 > y1:
            return img.crop((x1, y1, x2, y2))
        return img
    return Image.open(path_spec).convert('RGB')

def to_t(pil_img):
    return _T(pil_img).unsqueeze(0).to(DEVICE)

def tta_predict(pil_img):
    variants = [
        to_t(pil_img),
        to_t(TF.adjust_brightness(pil_img, 1.3)),
        to_t(TF.adjust_brightness(pil_img, 0.7)),
        to_t(TF.adjust_contrast(pil_img,   1.3)),
        to_t(TF.adjust_sharpness(pil_img,  2.0)),
    ]
    with torch.no_grad():
        avg = torch.stack([model_v4(v) for v in variants]).mean(0)
    return improved_ctc_decode(avg, CHARSET, BLANK_IDX)[0]

print('✓ Ready')

Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

✓ V4-CTC loaded  epoch=24  loss=0.6286
✓ Ready


In [10]:
# ── Cell 5: Sanity check — 5 samples from each existing dataset ───────────
from Levenshtein import distance as lev_distance

#print(f'{'GT':<15} {'PRED':<15} {'OK':>4}  dataset')
print('-'*50)

for name, txt in [('IIIT5K', IIIT5K_TXT), ('ArT', ART_TXT), ('TotalText', TOTALTEXT_TXT)]:
    if not os.path.exists(txt):
        print(f'  [{name}] file missing, skip')
        continue
    with open(txt) as f:
        lines = [l.strip() for l in f if l.strip()][:5]
    for line in lines:
        parts = line.split('\t')
        if len(parts) < 2: continue
        gt = ''.join(c for c in parts[-1].lower() if c in CHARSET_SET)
        if not gt: continue
        try:
            pil = load_img(parts[0].strip())
            with torch.no_grad():
                pred = improved_ctc_decode(model_v4(to_t(pil)), CHARSET, BLANK_IDX)[0]
            ok = '✓' if pred == gt else '✗'
            print(f'{gt:<15} {pred:<15} {ok:>4}  {name}')
        except Exception as e:
            print(f'  ERROR: {e}')

--------------------------------------------------
private         private            ✓  IIIT5K
parking         parking            ✓  IIIT5K
salutes         salutes            ✓  IIIT5K
dolce           dolce              ✓  IIIT5K
gabbana         gabbana            ✓  IIIT5K
1910            1910               ✓  ArT
despacho        despacho           ✓  ArT
hotel           hotel              ✓  ArT
shanghaibeicaiassetsmanagingcoltd saiecasesum        ✗  ArT
length          length             ✓  ArT
petrosains      setrosins          ✗  TotalText
the             the                ✓  TotalText
peak            peak               ✓  TotalText
lookout         lookout            ✓  TotalText
the             the                ✓  TotalText


In [11]:
# ── Cell 6: FULL EVALUATION — all 7 datasets ──────────────────────────────

def eval_dataset(txt_path, name, use_tta=False):
    if not os.path.exists(txt_path):
        print(f'  [{name}] ✗ file not found: {txt_path}')
        return None

    with open(txt_path, encoding='utf-8') as f:
        lines = [l.strip() for l in f if l.strip()]

    exact = off1 = off2 = total = 0
    ned_sum = 0.0
    t0 = time.time()

    with torch.no_grad():
        for i, line in enumerate(lines):
            parts = line.split('\t')
            if len(parts) < 2: continue
            gt = ''.join(c for c in parts[-1].strip().lower() if c in CHARSET_SET)
            if not gt: continue
            img_file = parts[0].strip().split('|')[0]
            if not os.path.exists(img_file): continue
            try:
                pil = load_img(parts[0].strip())
                if use_tta:
                    pred = tta_predict(pil)
                else:
                    pred = improved_ctc_decode(model_v4(to_t(pil)), CHARSET, BLANK_IDX)[0]
            except Exception:
                continue

            ed = lev_distance(pred, gt)
            ned_sum += ed / max(len(pred), len(gt), 1)
            if ed == 0: exact += 1
            if ed <= 1: off1  += 1
            if ed <= 2: off2  += 1
            total += 1

            if (i+1) % 500 == 0:
                rate = (i+1) / max(time.time()-t0, 0.01)
                eta  = (len(lines)-i-1) / rate
                tag = '+TTA' if use_tta else ''
                print(f'  {name}{tag}: {i+1}/{len(lines)}  '
                      f'exact={100*exact/total:.1f}%  ~{eta:.0f}s left', end='\r')

    if total == 0:
        print(f'  [{name}] ✗ no valid samples evaluated')
        return None

    r = dict(exact=100*exact/total, off1=100*off1/total,
             off2=100*off2/total,   ned=1-ned_sum/total, n=total)
    tag = ' +TTA' if use_tta else ''
    print(f'  [{name}{tag}]  n={total:,}  '
          f'exact={r["exact"]:.2f}%  '
          f'±1={r["off1"]:.2f}%  '
          f'±2={r["off2"]:.2f}%  '
          f'NED={r["ned"]:.4f}      ')
    return r


# Dataset registry
ALL_DATASETS = [
    ('IIIT5K',    'Regular',    IIIT5K_TXT),
    ('SVT',       'Regular',    SVT_TXT),
    ('IC13',      'Regular',    ICDAR13_TXT),
    ('IC15',      'Incidental', ICDAR15_TXT),
    ('ArT',       'Irregular',  ART_TXT),
    ('TotalText', 'Curved',     TOTALTEXT_TXT),
    ('SCUT',      'Curved',     SCUT_TXT),
]

USE_TTA = False  # Set True to also run TTA (doubles eval time)

print('='*65)
print('V4-CTC  FULL EVALUATION')
print('='*65)
results = {}
for name, dtype, txt in ALL_DATASETS:
    r = eval_dataset(txt, name, use_tta=False)
    if r:
        results[name] = r

if USE_TTA:
    print('\n' + '='*65)
    print('V4-CTC  FULL EVALUATION  +TTA')
    print('='*65)
    results_tta = {}
    for name, dtype, txt in ALL_DATASETS:
        r = eval_dataset(txt, name, use_tta=True)
        if r:
            results_tta[name] = r

V4-CTC  FULL EVALUATION
  [IIIT5K]  n=3,000  exact=89.87%  ±1=96.43%  ±2=98.43%  NED=0.9724      
  [SVT]  n=647  exact=82.07%  ±1=94.13%  ±2=98.61%  NED=0.9534      
  [IC13]  n=1,081  exact=84.55%  ±1=93.34%  ±2=97.13%  NED=0.9370      
  [IC15]  n=2,075  exact=68.92%  ±1=83.76%  ±2=91.37%  NED=0.8790      
  [ArT]  n=5,363  exact=70.78%  ±1=83.54%  ±2=89.71%  NED=0.8715      
  [TotalText]  n=2,211  exact=81.23%  ±1=90.00%  ±2=92.94%  NED=0.9242      


KeyboardInterrupt: 

In [12]:
# ── Cell 7: Print paper-ready results table ───────────────────────────────

TYPES = {
    'IIIT5K':'Regular', 'SVT':'Regular',     'IC13':'Regular',
    'IC15':'Incidental','ArT':'Irregular',   'TotalText':'Curved'
}

def print_table(res, title='V4-CTC'):
    ORDER = ['IIIT5K','SVT','IC13','IC15','ArT','TotalText']
    W = 78
    print('\n' + '='*W)
    print(f'  PAPER TABLE — PARSeq-GeoAware ({title})')
    print('='*W)
    print(f'  {"Dataset":<12} {"Type":<12} {"Exact":>8} {"±1 Char":>9} '
          f'{"±2 Char":>9} {"NED":>8} {"N":>7}')
    print('-'*W)

    reg_e = reg_n = irr_e = irr_n = 0
    for ds in ORDER:
        if ds not in res: continue
        r  = res[ds]
        tp = TYPES.get(ds, '')
        print(f'  {ds:<12} {tp:<12} {r["exact"]:>8.2f}%'
              f'{r["off1"]:>9.2f}% {r["off2"]:>9.2f}%'
              f'{r["ned"]:>8.4f} {r["n"]:>7,}')
        if tp == 'Regular':
            reg_e += r['exact']*r['n']; reg_n += r['n']
        else:
            irr_e += r['exact']*r['n']; irr_n += r['n']

    print('-'*W)
    if reg_n > 0:
        print(f'  {"Avg (Regular)":<24} {reg_e/reg_n:>8.2f}%')
    if irr_n > 0:
        print(f'  {"Avg (Irregular/Curved/Incidental)":<24} {irr_e/irr_n:>8.2f}%')
    print('='*W)

print_table(results, title='No TTA')

if USE_TTA and results_tta:
    print_table(results_tta, title='+TTA')

    # TTA gain table
    print('\n  TTA GAIN SUMMARY')
    print(f'  {"Dataset":<12} {"Base":>8} {"TTA":>8} {"Gain":>8}')
    print('  ' + '-'*38)
    for ds in results:
        if ds in results_tta:
            b = results[ds]['exact']
            t = results_tta[ds]['exact']
            print(f'  {ds:<12} {b:>8.2f}% {t:>8.2f}% {t-b:>+8.2f}%')


  PAPER TABLE — PARSeq-GeoAware (No TTA)
  Dataset      Type            Exact   ±1 Char   ±2 Char      NED       N
------------------------------------------------------------------------------
  IIIT5K       Regular         89.87%    96.43%     98.43%  0.9724   3,000
  SVT          Regular         82.07%    94.13%     98.61%  0.9534     647
  IC13         Regular         84.55%    93.34%     97.13%  0.9370   1,081
  IC15         Incidental      68.92%    83.76%     91.37%  0.8790   2,075
  ArT          Irregular       70.78%    83.54%     89.71%  0.8715   5,363
  TotalText    Curved          81.23%    90.00%     92.94%  0.9242   2,211
------------------------------------------------------------------------------
  Avg (Regular)               87.58%
  Avg (Irregular/Curved/Incidental)    72.77%


In [13]:
# ── Cell 8: Per-dataset error analysis ───────────────────────────────────
# Shows top-20 confusion pairs and length-wise accuracy for one dataset.
# Change ANALYSE_DATASET to inspect any benchmark.

from collections import Counter

ANALYSE_DATASET = 'ArT'   # change to 'IIIT5K', 'SVT', etc.

TXT_MAP = {name: txt for name, _, txt in ALL_DATASETS}
txt_path = TXT_MAP.get(ANALYSE_DATASET)

if txt_path and os.path.exists(txt_path):
    with open(txt_path, encoding='utf-8') as f:
        lines = [l.strip() for l in f if l.strip()]

    errors = []
    len_acc = Counter()
    len_tot = Counter()

    with torch.no_grad():
        for line in lines:
            parts = line.split('\t')
            if len(parts) < 2: continue
            gt = ''.join(c for c in parts[-1].strip().lower() if c in CHARSET_SET)
            if not gt: continue
            img_file = parts[0].strip().split('|')[0]
            if not os.path.exists(img_file): continue
            try:
                pil  = load_img(parts[0].strip())
                pred = improved_ctc_decode(model_v4(to_t(pil)), CHARSET, BLANK_IDX)[0]
            except:
                continue
            L = len(gt)
            len_tot[L] += 1
            if pred == gt:
                len_acc[L] += 1
            else:
                errors.append((gt, pred))

    # Top confusion pairs
    top_errors = Counter(errors).most_common(20)
    print(f'\nTop-20 errors on {ANALYSE_DATASET}:')
    print(f'  {"GT":<15} {"PRED":<15} {"Count":>6}')
    print('  ' + '-'*38)
    for (gt, pred), cnt in top_errors:
        print(f'  {gt:<15} {pred:<15} {cnt:>6}')

    # Accuracy by word length
    print(f'\nAccuracy by word length on {ANALYSE_DATASET}:')
    print(f'  {"Length":>8} {"Correct":>9} {"Total":>8} {"Acc":>8}')
    print('  ' + '-'*37)
    for L in sorted(len_tot):
        acc = 100*len_acc[L]/len_tot[L] if len_tot[L] else 0
        print(f'  {L:>8} {len_acc[L]:>9} {len_tot[L]:>8} {acc:>8.1f}%')


Top-20 errors on ArT:
  GT              PRED             Count
  --------------------------------------
  tuc             tc                   2
  opal            oprl                 2
  all             al                   2
  in              1n                   2
  lmages          images               2
  trafford        traford              2
  street          stret                2
  shanghaibeicaiassetsmanagingcoltd saiecasesum          1
  hamleys         kamleys              1
  pusat           pusht                1
  mouton          dltow                1
  nteed           eed                  1
  petrossian      ptresa               1
  piuzl           piuza                1
  c008            co0                  1
  world           orld                 1
  ttoo            oo                   1
  moulin          nmoow                1
  bugget          nuggen               1
  92632           72632                1

Accuracy by word length on ArT:
    Length   Correct    